<a href="https://colab.research.google.com/github/TalCordova/RS_Coller_TAU_26B/blob/master/notebook4_content_based_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Content-Based Filtering: Recommending by Item Features

So far we built **collaborative filtering** recommenders (item-item and user-user). They all read from the same place: the **utility matrix** of user–item interactions. Two movies are "similar" if *users rated them similarly*.

Content-based filtering keeps the **exact same machinery** — build a similarity between items, take the nearest ones, recommend *"Because you watched X…"* — but swaps out the **signal**:

| | Item-Item CF | Content-Based |
|---|---|---|
| Two items are similar if… | users rated them similarly | their **features** match |
| Signal source | the utility matrix (interactions) | item metadata (here: **genres**) |
| Needs interactions on the item? | **yes** | **no** |

That last row is the whole point. A brand-new movie with zero ratings is invisible to item-item CF — but it still has genres, so content-based can rank it immediately. (We'll come back to this when we discuss the cold-start problem.)

> **Scope:** this is the same item→item *"Because you watched X"* shape you already know, now driven by content. We keep preprocessing deliberately minimal — the goal is the *idea*, not a production feature pipeline.

---
> **How to use this notebook:** read the markdown, run each cell in order, and complete the 🏋️ exercise.

## Section 0: Setup

In [12]:
!pip install numpy pandas scikit-learn --quiet

In [13]:
import ast
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.neighbors import NearestNeighbors

pd.set_option('display.max_colwidth', 60)
print("✅ Packages loaded")

✅ Packages loaded


## Section 1: Load the Data

We use the **`movies_metadata.csv`** file from Kaggle's *The Movies Dataset* (metadata for ~45,000 movies, sourced from TMDB). Unlike the MovieLens ratings we used for collaborative filtering, this file is pure **item metadata**: no users, no interactions at all. That's exactly what we want: a different signal.

The file has been shared with you via Google Drive. Update the path below to match where you saved it in **your own Drive**.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

ModuleNotFoundError: No module named 'google'

In [ ]:
# Update this to wherever you placed the file in your Drive
METADATA_PATH = "/content/drive/YOUR_PATH_HERE/movies_metadata.csv"

# low_memory=False avoids the mixed-dtype warning this file is known for
movies = pd.read_csv(METADATA_PATH, low_memory=False)
print(f"Loaded {len(movies):,} movies")
movies[['title', 'genres']].head()

Look at the `genres` column above. It is **not** a clean list — each cell is a *stringified* list of dictionaries, e.g.

```
[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]
```

We only care about the genre **names**, so our one preprocessing job is to pull those out.

In [15]:
# If you run local
movies = pd.read_csv('movies_metadata.csv', low_memory=False)
movies.head()

,Unnamed: 0,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,...,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count,genres_tmp
0,0,FALSE,"{'id': 10194, 'name': 'Toy Story Collection', 'poster_pa...",30000000,"['Animation', 'Comedy', 'Family']",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,...,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0,"['Animation', 'Comedy', 'Family']"
1,1,FALSE,NaN,65000000,"['Adventure', 'Fantasy', 'Family']",NaN,8844,tt0113497,en,Jumanji,...,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso_639_1': '...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0,"['Adventure', 'Fantasy', 'Family']"
2,2,FALSE,"{'id': 119050, 'name': 'Grumpy Old Men Collection', 'pos...",0,"['Romance', 'Comedy']",NaN,15602,tt0113228,en,Grumpier Old Men,...,0.0,101.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Still Yelling. Still Fighting. Still Ready for Love.,Grumpier Old Men,False,6.5,92.0,"['Romance', 'Comedy']"
3,3,FALSE,NaN,16000000,"['Comedy', 'Drama', 'Romance']",NaN,31357,tt0114885,en,Waiting to Exhale,...,81452156.0,127.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Friends are the people who let you be yourself... and ne...,Waiting to Exhale,False,6.1,34.0,"['Comedy', 'Drama', 'Romance']"
4,4,FALSE,"{'id': 96871, 'name': 'Father of the Bride Collection', ...",0,['Comedy'],NaN,11862,tt0113041,en,Father of the Bride Part II,...,76578911.0,106.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,Just When His World Is Back To Normal... He's In For The...,Father of the Bride Part II,False,5.7,173.0,['Comedy']


## Section 2: Minimal Preprocessing — Genres → Multi-Hot

Two small steps:
1. **Parse** each `genres` string into a plain list of genre names (and quietly drop the handful of corrupt rows this file contains).
2. **Multi-hot encode**: turn the genre lists into a 0/1 matrix — one column per genre, a 1 where the movie has that genre.

In [16]:
def parse_genres(value):
    """Turn a stringified list-of-dicts into a list of genre names.
    Returns [] for the corrupt / malformed rows in this file."""
    try:
        parsed = ast.literal_eval(value)
        return [g['name'] if isinstance(g, dict) else g for g in parsed]
    except (ValueError, SyntaxError):
        return []

movies['genre_list'] = movies['genres'].apply(parse_genres)

# Keep only movies that actually have genres, and one row per title
before = len(movies)
movies = movies[movies['genre_list'].map(len) > 0]
movies = movies.drop_duplicates(subset='title').reset_index(drop=True)
print(f"Kept {len(movies):,} movies (dropped {before - len(movies):,} with no usable genres / duplicates)")
movies[['title', 'genre_list']].head()

Kept 40,034 movies (dropped 5,432 with no usable genres / duplicates)


,title,genre_list
0,Toy Story,"[Animation, Comedy, Family]"
1,Jumanji,"[Adventure, Fantasy, Family]"
2,Grumpier Old Men,"[Romance, Comedy]"
3,Waiting to Exhale,"[Comedy, Drama, Romance]"
4,Father of the Bride Part II,[Comedy]


In [17]:
# Multi-hot encode: one binary column per genre
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(movies['genre_list'])

print(f"Genre matrix shape: {genre_matrix.shape}  (movies × genres)")
print(f"Genres ({len(mlb.classes_)}): {', '.join(mlb.classes_)}")

# Peek: Toy Story as a row of 0s and 1s
example = movies.index[movies['title'] == 'Toy Story'][0]
pd.Series(genre_matrix[example], index=mlb.classes_, name='Toy Story').to_frame().T

Genre matrix shape: (40034, 20)  (movies × genres)
Genres (20): Action, Adventure, Animation, Comedy, Crime, Documentary, Drama, Family, Fantasy, Foreign, History, Horror, Music, Mystery, Romance, Science Fiction, TV Movie, Thriller, War, Western


,Action,Adventure,Animation,Comedy,Crime,Documentary,Drama,Family,Fantasy,Foreign,History,Horror,Music,Mystery,Romance,Science Fiction,TV Movie,Thriller,War,Western
Toy Story,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0


## Section 3: Build the Similarity

Each movie is now a vector in *genre space*. We measure how close two movies are with **cosine similarity** — the same metric we used for item-item CF. The only thing that changed is what the vectors are built from: **genres instead of ratings**.

We fit `NearestNeighbors` so we can look up the closest movies on demand, without ever materializing a giant 45,000 × 45,000 matrix.

In [18]:
nn = NearestNeighbors(metric='cosine', algorithm='brute')
nn.fit(genre_matrix)
print("✅ Similarity index ready")

✅ Similarity index ready


## Section 4: *"Because you watched X…"*

The recommendation step is identical in spirit to item-item CF: take a movie, find its nearest neighbors, return them.

In [19]:
title_to_index = pd.Series(movies.index, index=movies['title'])

def content_recommendations(title, k=10):
    """Return the k movies most similar to `title` by genre content."""
    if title not in title_to_index:
        raise KeyError(f"'{title}' not found. Check the exact title.")
    idx = title_to_index[title]

    distances, indices = nn.kneighbors([genre_matrix[idx]], n_neighbors=k + 1)
    similarities = 1 - distances.flatten()

    results = []
    for sim, i in zip(similarities, indices.flatten()):
        if i == idx:           # skip the movie itself
            continue
        results.append({'title': movies.loc[i, 'title'],
                        'genres': ', '.join(movies.loc[i, 'genre_list']),
                        'similarity': round(sim, 3)})
    return pd.DataFrame(results[:k])

In [20]:
print("Because you watched Inception:")
content_recommendations('Inception', k=10)

Because you watched Inception:


,title,genres,similarity
0,Paycheck,"Action, Adventure, Mystery, Science Fiction, Thriller",1.000
1,Sky Captain and the World of Tomorrow,"Mystery, Action, Thriller, Science Fiction, Adventure",1.000
2,Knowing,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
3,Congo,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
4,Zenith,"Action, Adventure, Drama, Mystery, Science Fiction, Thri...",0.913
5,Star Trek III: The Search for Spock,"Science Fiction, Action, Adventure, Thriller",0.894
6,The Abyss,"Adventure, Action, Thriller, Science Fiction",0.894
7,Bear Island,"Mystery, Adventure, Thriller, Action",0.894
8,Starship Troopers,"Adventure, Action, Thriller, Science Fiction",0.894
9,ISRA 88,"Thriller, Adventure, Science Fiction, Mystery",0.894


Look at the **scores**, not just the titles. Movies that share *all five* of Inception's genres come back at **1.0**, while ones that match *most but not all* land a notch lower (**0.913**, **0.894**). So the recommender genuinely **ranks by how much genre overlap there is** — it isn't just dumping a random pile of same-genre movies.

And notice it needed **no ratings at all** — only genres. We could recommend a movie that *nobody has ever watched*, as long as we know its genres. That is something collaborative filtering simply cannot do.

> Try a few **other** titles, though, and you'll often get a wall of identical **1.0**s. That's not a bug — it's the limitation you'll fix in the exercise below.

### 🏋️ Exercise: Break the 1.0 Ties

Run the cell below for *Toy Story*. Every neighbor comes back at **similarity = 1.0** — a wall of perfect scores.

**Why?** Cosine looks only at a vector's *direction*, and a multi-hot genre vector encodes nothing but *which* genres are present. Two movies with the **same genre set** have the **same vector**, so their cosine is exactly **1.0**. With only ~20 genres and most films carrying 1–3, this happens constantly: every `Animation, Comedy, Family` movie is identical to every other. Genres alone simply **cannot rank within a genre group**.

**Your task — give the vector something to break the ties with.** Add three cheap numeric features from the metadata and re-rank:

- **release year** — separates a 1995 comedy from a 2015 one
- **`vote_average`** — separates a well-loved film from a poorly-rated one
- **`vote_count`** — a popularity signal: blockbuster vs. obscure title

Steps in the 🏋️ cell:
1. Pull the three columns (log-scale `vote_count` — it's heavily skewed — and fill missing values with the median).
2. **Scale them to [0, 1]** with `MinMaxScaler` so they sit on the same scale as the 0/1 genre bits. *(Skip this and a raw `vote_count` in the thousands would swamp every genre.)*
3. `np.hstack` them onto `genre_matrix`, fit a fresh `NearestNeighbors`, and recommend again.

Then compare: did the ties break? Are the new neighbors actually *better*?

In [21]:
# BEFORE — genres only: a wall of 1.0s
content_recommendations('Toy Story', k=10)

,title,genres,similarity
0,Hoodwinked!,"Animation, Comedy, Family",1.0
1,How the Grinch Stole Christmas!,"Animation, Family, Comedy",1.0
2,Meet the Robinsons,"Animation, Comedy, Family",1.0
3,Superstar Goofy,"Animation, Comedy, Family",1.0
4,Saving Santa,"Animation, Comedy, Family",1.0
5,Animals United,"Animation, Family, Comedy",1.0
6,Garfield: A Tail of Two Kitties,"Animation, Comedy, Family",1.0
7,Over the Hedge,"Comedy, Animation, Family",1.0
8,Tom and Jerry: The Lost Dragon,"Animation, Comedy, Family",1.0
9,"The Looney, Looney, Looney Bugs Bunny Movie","Animation, Comedy, Family",1.0


In [22]:
# 1. Three numeric features (log-scale the skewed vote_count, fill gaps with the median)
from sklearn.preprocessing import MinMaxScaler

feats = pd.DataFrame({
    'year':         movies['release_date'].astype(str).str.extract(r'(\d{4})')[0].astype(float),
    'vote_average': pd.to_numeric(movies['vote_average'], errors='coerce'),
    'vote_count':   np.log1p(pd.to_numeric(movies['vote_count'], errors='coerce')),
})
feats = feats.fillna(feats.median())

# 2. Scale to [0, 1] so the features share the genre bits' scale, then 3. stack onto the genres
feats_scaled = MinMaxScaler().fit_transform(feats)
enriched_matrix = np.hstack([genre_matrix, feats_scaled])

# A fresh index, built on the richer vectors
nn_enriched = NearestNeighbors(metric='cosine', algorithm='brute').fit(enriched_matrix)

def content_recommendations_enriched(title, k=10):
    # Same logic as before, on the enriched matrix; we also surface `year` to eyeball the change
    idx = title_to_index[title]
    distances, indices = nn_enriched.kneighbors([enriched_matrix[idx]], n_neighbors=k + 1)
    similarities = 1 - distances.flatten()
    results = []
    for sim, i in zip(similarities, indices.flatten()):
        if i == idx:
            continue
        results.append({'title': movies.loc[i, 'title'],
                        'genres': ', '.join(movies.loc[i, 'genre_list']),
                        'year': int(feats.loc[i, 'year']),
                        'similarity': round(sim, 3)})
    return pd.DataFrame(results[:k])

print("AFTER — Toy Story (genres + year + votes):\n")
display(content_recommendations_enriched('Toy Story', k=10))

print("\nAFTER — GoldenEye:\n")
display(content_recommendations_enriched('GoldenEye', k=10))

AFTER — Toy Story (genres + year + votes):



,title,genres,year,similarity
0,"Monsters, Inc.","Animation, Comedy, Family",2001,1.000
1,Toy Story 2,"Animation, Comedy, Family",1999,1.000
2,Toy Story 3,"Animation, Family, Comedy",2010,0.999
3,Despicable Me 2,"Animation, Comedy, Family",2013,0.998
4,The Simpsons Movie,"Animation, Comedy, Family",2007,0.998
5,Chicken Run,"Animation, Comedy, Family",2000,0.997
6,Cloudy with a Chance of Meatballs,"Animation, Comedy, Family",2009,0.996
7,Hotel Transylvania 2,"Animation, Comedy, Family",2015,0.995
8,Over the Hedge,"Comedy, Animation, Family",2006,0.995
9,Meet the Robinsons,"Animation, Comedy, Family",2007,0.995



AFTER — GoldenEye:



,title,genres,year,similarity
0,The Rock,"Action, Adventure, Thriller",1996,1.000
1,Tomorrow Never Dies,"Adventure, Action, Thriller",1997,1.000
2,The World Is Not Enough,"Adventure, Action, Thriller",1999,0.999
3,The Hunt for Red October,"Action, Adventure, Thriller",1990,0.999
4,Commando,"Action, Adventure, Thriller",1985,0.999
5,Mission: Impossible,"Adventure, Action, Thriller",1996,0.999
6,Cliffhanger,"Action, Adventure, Thriller",1993,0.999
7,Mission: Impossible III,"Adventure, Action, Thriller",2006,0.999
8,Mission: Impossible II,"Adventure, Action, Thriller",2000,0.999
9,Die Another Day,"Adventure, Action, Thriller",2002,0.999


### 🔑 Solution

<details>
<summary>🔑 Reveal explanation</summary>

**Why the wall of 1.0s?** Cosine compares *direction* only, and a multi-hot genre vector encodes nothing but which genres are present — so two movies with an identical genre set are the *same vector*, and their similarity is exactly 1.0. With ~20 genres and most films carrying 1–3, near-duplicates are everywhere.

**What the three features add.** `year`, `vote_average` and `log(vote_count)` give movies that share a genre set something to differ on. The `MinMaxScaler` step is the one that matters: the genre columns are 0/1, so the numeric features must be squeezed into the same [0, 1] range — otherwise a raw `vote_count` in the thousands dominates the cosine and the genres stop mattering.

**The payoff.** *Toy Story*'s neighbors go from a random pile of obscure cartoons to **Monsters, Inc.**, **Toy Story 2/3**, **Despicable Me 2** — same genres, *and* the same era + popularity tier. *GoldenEye* goes from direct-to-video filler to **The Rock**, **Tomorrow Never Dies**, **The Hunt for Red October**.

**Why are the scores still ~1.0?** The vector is 23 dimensions, 20 of them genres, so genres still dominate — the numeric features act as a *tie-breaker within* a genre cluster rather than reshaping the whole space. That is exactly what we wanted here. To make the scores themselves spread out, weight the numeric block up (e.g. `feats_scaled * 3`) before stacking.

</details>

## Section 5: Where This Fits

**Content-based is another signal source, not a better algorithm.** It sits alongside item-item and user-user CF: same retrieval machinery, different input.

- **vs. CF:** CF needs interactions; content does not. CF can capture taste patterns genres miss ("people who like *this* obscure film also like…"); content can recommend brand-new or never-watched items. They are complementary — real systems blend them.
- **Cold start (next topic):** because content similarity comes from features, it handles the **item** cold-start case directly — a new movie is rankable the moment we know its genres. Note this fixes the *item* side only: a brand-new **user** with no history still has no profile to match against.

### Going further (not needed for this tutorial)
Genres are a coarse signal — in the exercise you broke the worst ties by adding a few numeric features (year, votes). The next upgrade is richer **text** features: the `overview`, cast/crew, keywords. Encode the `overview` with TF-IDF or sentence embeddings (e.g. a BERT-family model) for a much finer-grained similarity. Same machinery — just a richer item vector.